# Kustomize 04: Helm charts inside Kustomize, validation, GitOps

`helmCharts` renders a chart into the overlay (needs `--enable-helm`, and `kustomize.buildOptions` in Argo CD). Validation is external: kubeconform, server-side dry run, or a CUE schema.


In [ ]:
cd /source/work/kustomize-lab/overlays/prod
export HOME=/tmp
cat >> kustomization.yaml <<'YAML'
helmCharts:
  - name: podinfo
    repo: https://stefanprodan.github.io/podinfo
    version: 6.15.0
    releaseName: podinfo
    namespace: web-prod
    skipTests: true
    valuesInline:
      replicaCount: 2
YAML
kustomize build --enable-helm . | yq '.kind + "/" + .metadata.name' | sort


In [ ]:
cd /source/work/kustomize-lab
kustomize build --enable-helm overlays/prod > /tmp/prod.yaml && kubeconform -strict -ignore-missing-schemas -summary /tmp/prod.yaml


In [ ]:
cd /source/work/kustomize-lab
kustomize build overlays/dev > /tmp/dev.yaml; diff <(yq '.kind + "/" + .metadata.name' /tmp/dev.yaml | sort) <(yq '.kind + "/" + .metadata.name' /tmp/prod.yaml | sort) || true


The companion repository commits the build output under `rendered/kustomize/` and lets Argo CD apply a directory: reviewable diffs, no render at sync time. The native alternative is an Argo CD Application pointing at the overlay, or a Flux Kustomization.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers && sed -n 1,25p gitops/argocd/web-kustomize-prod.yaml


Try it: bump the podinfo chart one minor version and list which objects in the build change.
